# 07. Headline Statistics

[![Repo](https://img.shields.io/badge/GitHub-metaheuristic--budget--reproduction-181717?logo=github&logoColor=white)](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) [![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE) [![Author](https://img.shields.io/badge/author-Prakash%20Ukhalkar-blue?logo=orcid&logoColor=white)](https://orcid.org/0000-0002-0452-6574) [![Python](https://img.shields.io/badge/python-3.10%2B-blue)](../requirements.txt)

**Source script:** `src/extra_analysis.py` &nbsp;|&nbsp; **Notebook 7 of 10**

Computes the headline statistics reported in the revised manuscript.

Part of *Budget-Controlled Reproduction Study of Nature-Inspired Metaheuristics* — a reproduction study comparing six metaphor-based metaheuristics (GWO, WOA, SCA, SSA, HHO, AOA) against five established baselines (DE, PSO, L-SHADE, CMA-ES, random search) on constrained engineering design problems, under matched evaluation budgets and tuning effort.

See the [repository README](../README.md) for installation and full reproduction instructions, and [notebooks/README.md](README.md) for the notebook index and suggested run order.

---


# Headline Statistics for the Revised Manuscript

![Python](https://img.shields.io/badge/python-3.10%2B-blue) ![Status](https://img.shields.io/badge/status-research--reproduction-lightgrey) ![License](https://img.shields.io/badge/license-MIT-green)

Aggregates fidelity, budget-sensitivity, tuning and constraint-scheme results into a single headline summary (`headline_v2.json`) used for the revised manuscript text.


## Imports

External libraries and project modules used by this notebook.

In [ ]:
import json, os, numpy as np, pandas as pd
from analysis import load, ORDER, TEST_GROUP, BASELINES, TRAIN_PROBLEMS, friedman_nemenyi, pairwise_wilcoxon

In [ ]:
OUT = os.path.join(os.path.dirname(__file__), "..", "results")
res = {}

dm, dt = load("main"), load("tuned")
allp = list(dm.problem.unique())
scale = [p for p in allp if p != "GearTrain"]
ho = [p for p in scale if p not in TRAIN_PROBLEMS]

_, st, pv, ranks, cd = friedman_nemenyi(dm, allp)
R = pairwise_wilcoxon(dm, allp)
res["friedman"] = dict(stat=float(st), p=float(pv), cd=float(cd), ranks=ranks.to_dict())
res["wtl_total"] = R.outcome.value_counts().to_dict()
res["median_A12"] = float(R.A12.median())
res["wtl_vs"] = {b: R[R.base == b].outcome.value_counts().to_dict() for b in BASELINES}
res["wtl_by_alg_vs_CMAES"] = {a: R[(R.test==a)&(R.base=="CMAES")].outcome.value_counts().to_dict() for a in TEST_GROUP}
res["wtl_by_alg_vs_RS"] = {a: R[(R.test==a)&(R.base=="RS")].outcome.value_counts().to_dict() for a in TEST_GROUP}
res["hit_rate"] = (dm[dm.problem.isin(scale)].assign(h=lambda d: d.gap<=1e-4)
                   .groupby("algorithm")["h"].mean().reindex(ORDER).to_dict())
res["feasibility"] = dm.groupby("algorithm")["feasible"].mean().to_dict()
med = dm[dm.problem.isin(scale)].groupby(["algorithm","problem"])["gap"].median().unstack()
tun = dt[dt.problem.isin(scale)].groupby(["algorithm","problem"])["gap"].median().unstack()
res["tuning"] = {a: dict(default=float(med.loc[a, ho].mean()), tuned=float(tun.loc[a, ho].mean()))
                 for a in ORDER}
res["n_runs"] = int(dm.groupby(["algorithm","problem"]).size().max())

for sc in ("static", "eps"):
    d = load("cons", sc); cp = list(d.problem.unique())
    _, _, _, r, _ = friedman_nemenyi(d, cp)
    res.setdefault("rank_by_scheme", {})[sc] = r.to_dict()
cp = list(load("cons","static").problem.unique())
_, _, _, rdeb, _ = friedman_nemenyi(dm[dm.problem.isin(cp)], cp)
res["rank_by_scheme"]["deb"] = rdeb.to_dict()

for b in (5000, 50000):
    f = f"{OUT}/budget{b}_deb.json"
    if os.path.exists(f):
        d = load(f"budget{b}")
        bp = list(d.problem.unique())
        _, _, _, r, _ = friedman_nemenyi(d, bp)
        res.setdefault("rank_by_budget", {})[str(b)] = r.to_dict()
_, _, _, r15, _ = friedman_nemenyi(dm[dm.problem.isin(["WeldedBeam","SpeedReducer","PressureVessel","TensionSpring"])],
                                   ["WeldedBeam","SpeedReducer","PressureVessel","TensionSpring"])
res.setdefault("rank_by_budget", {})["15000"] = r15.to_dict()

if os.path.exists(f"{OUT}/fidelity.json"):
    F = pd.DataFrame(json.load(open(f"{OUT}/fidelity.json")))
    res["fidelity"] = dict(
        n_pairs=int(len(F)),
        n_agree=int((~(F.p_ranksum < 0.05)).sum()),
        median_rel_diff=float(F.rel_diff.median()),
        by_alg={a: dict(agree=int((~(F[F.algorithm==a].p_ranksum<0.05)).sum()),
                        n=int((F.algorithm==a).sum()),
                        median_rel_diff=float(F[F.algorithm==a].rel_diff.median()))
                for a in F.algorithm.unique()},
        reference_nfe_mean={a: float(F[F.algorithm==a].mean_nfe_reference.mean())
                            for a in F.algorithm.unique()})
json.dump(res, open(f"{OUT}/headline_v2.json","w"), indent=1, default=float)
print(json.dumps(res, indent=1, default=float)[:2500])

---
## Key outcomes

- **Hit rate** (fraction of runs landing within 1e-4 relative gap of the published optimum, scale
  problems): DE (0.82) and L-SHADE (0.80) hit the published optimum in the vast majority of runs;
  among metaphor-based methods GWO and WOA are best (0.36 each), while SCA, AOA and RS essentially
  never do (<=0.014).
- **Tuning effect** (default vs. matched-budget tuned, held-out problems): tuning helps most algorithms
  modestly (e.g. WOA improves from 0.026 to 0.011 mean gap; PSO from 0.042 to 0.010), but **hurts SSA**
  (0.081 -> 0.113), consistent with the overfitting-to-training-set caveat raised during the tuning phase.
- **Budget sensitivity** (5,000 / 15,000 / 50,000 evaluations): DE and CMA-ES occupy the top 2-3 ranks at
  every budget; HHO is last at every budget, confirming the ranking is not an artefact of the specific
  15,000-evaluation budget used for the headline results.
- Implementation-fidelity cross-check (from notebook 05) is folded in here: 19/48 algorithm-problem pairs
  show no significant difference from the independent `mealpy` implementation.

*Part of the budget-controlled reproduction study of nature-inspired metaheuristics.*


---

**Author:** Prakash Ukhalkar ([ORCID: 0000-0002-0452-6574](https://orcid.org/0000-0002-0452-6574)) — Pimpri Chinchwad College of Engineering, Pune, India

**Repository:** [github.com/prakash-ukhalkar/metaheuristic-budget-reproduction](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) &nbsp;|&nbsp; **License:** [MIT](../LICENSE) &nbsp;|&nbsp; **Citation:** [CITATION.cff](../CITATION.cff)

[![Repo](https://img.shields.io/badge/GitHub-metaheuristic--budget--reproduction-181717?logo=github&logoColor=white)](https://github.com/prakash-ukhalkar/metaheuristic-budget-reproduction) [![License](https://img.shields.io/badge/license-MIT-green)](../LICENSE)
